<a href="https://colab.research.google.com/github/hajyhia/Airbnb_Berlin_Price_predic/blob/main/06_Airbnb_Berlin_Model_Selection_Finetuning_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install missingno
# !pip install geopy
# !pip install xgboost

In [2]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
# from pandas_profiling import ProfileReport
import numpy as np
import missingno as msno
sns.set()
# plt.style.use('ggplot')
plt.style.use('seaborn-v0_8')
import warnings
import datetime as dt

from sklearn import ensemble, tree, linear_model
from sklearn import tree
from sklearn.metrics import accuracy_score
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.impute import KNNImputer

from scipy.stats import pearsonr
from scipy.stats import ks_2samp
from scipy.stats import norm
from scipy import stats
from scipy.stats import chisquare
from scipy.stats import chi2_contingency
from scipy.stats import f_oneway

# Ignore warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
df_Feature_Engineering = pd.read_pickle('/content/drive/My Drive/Airbnb/df_Feature_Engineering.pkl')

# Feature Selection

In [5]:
df_Feature_Selection = pd.read_pickle('/content/drive/My Drive/Airbnb/df_Feature_Selection.pkl')
df_Feature_Selection.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23536 entries, 0 to 23535
Data columns (total 34 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Accuracy Rating               23536 non-null  float64       
 1   Bathrooms                     23536 non-null  float64       
 2   Checkin Rating                23536 non-null  float64       
 3   Cleanliness Rating            23536 non-null  float64       
 4   Communication Rating          23536 non-null  float64       
 5   Latitude                      23536 non-null  float64       
 6   Location Rating               23536 non-null  float64       
 7   Longitude                     23536 non-null  float64       
 8   Overall Rating                23536 non-null  float64       
 9   Price                         23536 non-null  float64       
 10  Value Rating                  23536 non-null  float64       
 11  Comments                    

In [6]:
df_Feature_Selection['Distance From Center Reduced']

,Distance From Center Reduced
0,km_4
1,km_2
2,km_4
3,km_2
4,km_4
...,...
23531,km_25
23532,km_25
23533,km_4
23534,km_25


In [7]:
df = df_Feature_Selection.copy()

In [8]:
df = df.drop(columns=['Host Since','Host Since Year', 'Postal Code', 'Property Type', 'Comments','neighbourhood',
                      'Longitude', 'Latitude'], inplace=False)

# ,
#                       'Distance From Center', 'Longitude', 'Latitude'

In [9]:
df = df[(df['Price'] <= 600) & (df['Price'] > 0)]

## One-Hot Enciding and Label Encoding

In [10]:
from sklearn.preprocessing import OrdinalEncoder

df_object =  df.select_dtypes(include = ['object','category']).columns
for col in df_object:
  ord_enc = OrdinalEncoder()
  df[[col]] = ord_enc.fit_transform(df[[col]]).astype('int')

In [11]:
df.head(2)

,Accuracy Rating,Bathrooms,Checkin Rating,Cleanliness Rating,Communication Rating,Location Rating,Overall Rating,Price,Value Rating,Host Response Time,Is Superhost,Neighborhood Group,Is Exact Location,Room Type,Instant Bookable,Accomodates,Bedrooms,Beds,Guests Included,Min Nights,Reviews,Property Type Reduced,Postal Code Reduced,Distance From Center,Distance From Center Reduced,Host Since From Now
0,10.0,1.0,10.0,10.0,10.0,9.0,100.0,17.0,10.0,2,False,6,True,1,False,2.0,1.0,1.0,1.0,2.000000,7.0,0,0,5.1,7,17
1,9.0,1.0,9.0,9.0,9.0,10.0,92.0,90.0,9.0,2,False,6,True,0,False,4.0,1.0,2.0,2.0,1.727273,144.0,0,0,3.7,5,17


## Multivariable Analysis

In [12]:
# Creating Variables dataframeS
varSel = pd.DataFrame({'Variable': df.columns.drop('Price')})
varSel

,Variable
0,Accuracy Rating
1,Bathrooms
2,Checkin Rating
3,Cleanliness Rating
4,Communication Rating
5,Location Rating
6,Overall Rating
7,Value Rating
8,Host Response Time
9,Is Superhost


In [13]:
df.to_csv('/content/drive/My Drive/Airbnb/df_Feature_Selection.csv')

In [14]:
nm = df.columns.drop('Price')
nm = nm.append(pd.Index(['Price']))
nm

Index(['Accuracy Rating', 'Bathrooms', 'Checkin Rating', 'Cleanliness Rating',
       'Communication Rating', 'Location Rating', 'Overall Rating',
       'Value Rating', 'Host Response Time', 'Is Superhost',
       'Neighborhood Group', 'Is Exact Location', 'Room Type',
       'Instant Bookable', 'Accomodates', 'Bedrooms', 'Beds',
       'Guests Included', 'Min Nights', 'Reviews', 'Property Type Reduced',
       'Postal Code Reduced', 'Distance From Center',
       'Distance From Center Reduced', 'Host Since From Now', 'Price'],
      dtype='object')

In [15]:
# df2 = df[nm].copy()
# df2 = df2.dropna()

In [16]:
X = df.drop(columns='Price', inplace=False)
y = df['Price']

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso
from sklearn.feature_selection import SelectFromModel
from sklearn.svm import LinearSVR
from sklearn.linear_model import Ridge


### Variable Selection using LASSO (L1 penalization)

In [18]:
lassomod = Lasso(alpha=0.01).fit(X, y)
model = SelectFromModel(lassomod, prefit=True)
# print(f"get_support: {model.get_support()}")
# print(f"coef: {lassomod.coef_}")
varSel['Lasso'] = model.get_support().astype('int64')

### Variable Selection using Ridge

In [19]:
ridge = Ridge(alpha=5).fit(X, y)
model = SelectFromModel(ridge, prefit=True)
# print(f"get_support: {model.get_support()}")
# print(f"coef: {ridge.coef_}")
varSel['Ridge'] = model.get_support().astype('int64')

### Variable Selection using Gradient Boosting classification

In [20]:
gbmod = GradientBoostingRegressor().fit(X, y)
model = SelectFromModel(gbmod , prefit=True)
# print(f"get_support: {model.get_support()}")
# print(f"coef: {gbmod.coef_}")
varSel['GradientBoost'] = model.get_support().astype('int64')

### Variable Selection using Random Forest

In [21]:
rfmod = RandomForestRegressor().fit(X, y)
model = SelectFromModel(rfmod, prefit=True)
# print(f"get_support: {model.get_support()}")
# print(f"coef: {rfmod.coef_}")
varSel['RandomForest'] = model.get_support().astype('int64')

### Summarization and Selection of Variables

In [22]:
varSel['Sum'] = varSel[['Lasso', 'Ridge', 'GradientBoost','RandomForest']].sum(axis=1)
varSel

,Variable,Lasso,Ridge,GradientBoost,RandomForest,Sum
0,Accuracy Rating,1,0,0,0,1
1,Bathrooms,0,0,0,0,0
2,Checkin Rating,0,0,0,0,0
3,Cleanliness Rating,1,1,0,0,2
4,Communication Rating,1,0,0,0,1
5,Location Rating,1,1,0,0,2
6,Overall Rating,1,0,0,0,1
7,Value Rating,1,1,0,0,2
8,Host Response Time,1,0,0,0,1
9,Is Superhost,1,0,0,0,1


In [23]:
varSel.to_csv('/content/drive/My Drive/Airbnb/Feature_Selection_models.csv')

In [24]:
varSel[['Variable', 'Sum']].sort_values(by='Sum', ascending=False)

,Variable,Sum
12,Room Type,4
17,Guests Included,4
15,Bedrooms,4
14,Accomodates,4
20,Property Type Reduced,3
7,Value Rating,2
5,Location Rating,2
19,Reviews,2
3,Cleanliness Rating,2
24,Host Since From Now,2


In [25]:
varSel[varSel['Sum'] >=2].shape

(12, 6)

### Creating DataFrame with most valuable variables

In [26]:
selected_vars = varSel[varSel['Sum'] >=2]['Variable']

In [27]:
df_model_ = df.loc[:,selected_vars]
df_model_['Price'] = df['Price'].copy()
# Output the result to verify
df_model_.info()

<class 'pandas.core.frame.DataFrame'>
Index: 23460 entries, 0 to 23535
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Cleanliness Rating     23460 non-null  float64
 1   Location Rating        23460 non-null  float64
 2   Value Rating           23460 non-null  float64
 3   Room Type              23460 non-null  int64  
 4   Accomodates            23460 non-null  float64
 5   Bedrooms               23460 non-null  float64
 6   Beds                   23460 non-null  float64
 7   Guests Included        23460 non-null  float64
 8   Reviews                23460 non-null  float64
 9   Property Type Reduced  23460 non-null  int64  
 10  Distance From Center   23460 non-null  float64
 11  Host Since From Now    23460 non-null  int64  
 12  Price                  23460 non-null  float64
dtypes: float64(10), int64(3)
memory usage: 2.5 MB


## Multivariable Analysis

In [ ]:
# Fit models and determine if a feature is selected (1) or not (0)
lasso = Lasso(alpha=5).fit(X, y)
lasso_selected = (np.abs(lasso.coef_) > 0).astype(int)

# Fit Ridge model
ridge = Ridge(alpha=5).fit(X, y)
ridge_selected = (np.abs(ridge.coef_) > 0).astype(int)

gb = GradientBoostingRegressor().fit(X, y)
gb_selected = (gb.feature_importances_ > 0).astype(int)

rf = RandomForestRegressor().fit(X, y)
rf_selected = (rf.feature_importances_ > 0).astype(int)

# Create a DataFrame to store results
selection_df = pd.DataFrame({
    'Feature': X.columns,
    'Lasso': lasso_selected,
    'GradientBoost': gb_selected,
    'RandomForest': rf_selected,
    'Ridge': ridge_selected
})

# Sum the number of selections for each feature
selection_df['Sum'] = selection_df[['Lasso', 'GradientBoost', 'RandomForest', 'Ridge']].sum(axis=1)

# Output the results
selection_df

In [ ]:
selection_df.to_csv('/content/selection_variable_modele.csv')

In [ ]:
selection_df.shape

In [ ]:
selection_df[['Feature', 'Sum']].sort_values(by='Sum', ascending=False)

In [ ]:
selection_df[selection_df['Sum'] >= 3].shape

In [ ]:
#Selecting variables with a sum of selections >= 4
selected_variables = selection_df[selection_df['Sum'] >= 3]['Feature']
selected_variables

### Creating DataFrame with most valuable variables

In [ ]:
df_model = df.loc[:,selected_variables]
df_model['Price'] = df['Price'].copy()

# Output the result to verify
df_model.info()

## Store Objects States

In [ ]:
selection_df.to_pickle("/content/drive/My Drive/Airbnb/feature_selection_df.pkl")
df_model.to_pickle("/content/drive/My Drive/Airbnb/df_Model_Selection.pkl")

# Model Selection and Fine Tuning

## Metrics

In [ ]:
df_Model_Selection = pd.read_pickle('/content/drive/My Drive/Airbnb/df_Model_Selection.pkl')
df_Model_Selection.info()

In [ ]:
df = df_Model_Selection.copy()
df = df[(df['Price'] <= 600) & (df['Price'] > 0)]

We reached this stage with only the most important feature bases on 3 out of 4 models (Review Feature Selection stage )

In [ ]:
df.head()

In [ ]:
df.describe().T

In [ ]:
df.corr()

In [ ]:
plt.figure(figsize=(15,10))
sns.heatmap(df.corr(), annot=True, fmt=".2f", annot_kws={"size":10})
plt.show()

## Create and Train the Model

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
import xgboost as xgb
import sklearn.metrics as metrics

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns='Price', inplace=False)
y = df['Price']

# Split into train+val and test sets (80% train+val, 20% test)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Split train+val into train and val sets (75% train, 25% val from the train+val set)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42
)

## Regression Models

In [ ]:
models_list = pd.DataFrame(columns= ['model', 'MAE', 'MSE', 'RMSE', 'RMSLE', 'R²'])
models_list.style.set_properties(subset=['model'], **{'text-align': 'left'})

In [ ]:
def regressionMetrics(y, y_pred):
    res = {
          'MAE': metrics.mean_absolute_error(y, y_pred),
          'MSE': metrics.mean_squared_error(y, y_pred),
          'RMSE': np.sqrt(metrics.mean_squared_error(y, y_pred)),
          'RMSLE': np.sqrt(metrics.mean_squared_log_error(np.abs(y), np.abs(y_pred))),
          'R²': metrics.r2_score(y, y_pred)
          }

    return res

### Linear Regression

In [ ]:
# create and train model
model_LR = LinearRegression()
model_LR.fit(X_train, y_train)

In [ ]:
# make predictions on the validation set
y_val_pred_LR = model_LR.predict(X_val)

In [ ]:
regressionMetrics(y_val, y_val_pred_LR)

In [ ]:
model_dict = {"model":"LinearRegression"}
model_dict.update(regressionMetrics(y_val, y_val_pred_LR))
model_dict
new_row = pd.DataFrame([model_dict])
models_list = pd.concat([models_list, new_row], ignore_index=True)

In [ ]:
models_list

In [ ]:
sns.scatterplot(x=y_val, y=y_val_pred_LR)

### *Decision Tree Regressor*

In [ ]:
model_DTR = DecisionTreeRegressor(random_state=42)
model_DTR.fit(X_train, y_train)

In [ ]:
# make predictions on the validation set
y_val_pred_DTR = model_DTR.predict(X_val)

In [ ]:
regressionMetrics(y_val, y_val_pred_DTR)

In [ ]:
model_dict = {"model":"DecisionTree"}
model_dict.update(regressionMetrics(y_val, y_val_pred_DTR))
new_row = pd.DataFrame([model_dict])
models_list = pd.concat([models_list, new_row], ignore_index=True)
# models_list.style.set_properties(**{'text-align': 'left'})
models_list

In [ ]:
sns.scatterplot(x=y_val, y=y_val_pred_DTR)

### RandomForestRegressor

In [ ]:
model_RFR = RandomForestRegressor(random_state=42)
model_RFR.fit(X_train, y_train)

In [ ]:
# Make predictions on the validation set
y_val_pred_RFR = model_RFR.predict(X_val)

In [ ]:
regressionMetrics(y_val, y_val_pred_RFR)

In [ ]:
model_dict = {'model': "RandomForest"}
new_row = pd.DataFrame([{**model_dict, **regressionMetrics(y_val, y_val_pred_RFR)}])
models_list = pd.concat([models_list, new_row], ignore_index=True)
models_list

In [ ]:
sns.scatterplot(x=y_val, y=y_val_pred_RFR)

### Adaptive Boosting (ADABoost)

In [ ]:
model_ADABoost = AdaBoostRegressor(random_state=42)
model_ADABoost.fit(X_train, y_train)

In [ ]:
# Make predictions on the validation set
y_val_pred_ADABoost = model_ADABoost.predict(X_val)

In [ ]:
regressionMetrics(y_val, y_val_pred_ADABoost)

Update model list with **Adaptive Boosting (ADABoost)** model results

In [ ]:
model_dict = {'model': "ADABoost"}
new_row = pd.DataFrame([{**model_dict, **regressionMetrics(y_val, y_val_pred_ADABoost)}])
models_list = pd.concat([models_list, new_row], ignore_index=True)
models_list

In [ ]:
sns.scatterplot(x=y_val, y=y_val_pred_ADABoost)

### Gradient Boosting Machine (GBM)

In [ ]:
model_GBM = GradientBoostingRegressor(random_state=42)
model_GBM.fit(X_train, y_train)

In [ ]:
# Make predictions on the validation set
y_val_pred_GBM = model_GBM.predict(X_val)

In [ ]:
model_dict = {'model': "GBM"}
new_row = pd.DataFrame([{**model_dict, **regressionMetrics(y_val, y_val_pred_GBM)}])
models_list = pd.concat([models_list, new_row], ignore_index=True)
models_list

In [ ]:
sns.scatterplot(x=y_val, y=y_val_pred_GBM)

### Support Vector Machine (SVM)

In [ ]:
model_SVR = SVR()
model_SVR.fit(X_train, y_train)

In [ ]:
# Make predictions on the validation set
y_val_pred_SVR = model_SVR.predict(X_val)

In [ ]:
regressionMetrics(y_val,y_val_pred_SVR)

In [ ]:
model_dict = {'model': "SVM"}
new_row = pd.DataFrame([{**model_dict, **regressionMetrics(y_val,y_val_pred_SVR)}])
models_list = pd.concat([models_list, new_row], ignore_index=True)
models_list

In [ ]:
sns.scatterplot(x=y_val, y=y_val_pred_SVR)

### XGBoost Regressor

In [ ]:
model_XGBoost= xgb.XGBRegressor()
model_XGBoost.fit(X_train, y_train)
# Make predictions on the validation set
y_val_pred_XGBoost = model_XGBoost.predict(X_val)

In [ ]:
regressionMetrics(y_val,y_val_pred_XGBoost)

In [ ]:
model_dict = {'model': "XGBoost"}
new_row = pd.DataFrame([{**model_dict, **regressionMetrics(y_val,y_val_pred_XGBoost)}])
models_list = pd.concat([models_list, new_row], ignore_index=True)
models_list

In [ ]:
sns.scatterplot(x=y_val, y=y_val_pred_XGBoost)

Metrics:<br><b>MSE</b> - Mean Squared Error<br><b>RMSE</b> Root Mean Squared Error<br><b>MAE </b>Mean Absolute Error Calculates the average of the absolute differences between predicted and actual values.<br>
<b>RMSLE</b> Root Mean Squared Logarithmic Error

In [ ]:
models_list.sort_values('MAE')

## Hyperparameters and Finetuning

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV

### Random Search: we decide which parameters and how (randomly)

In [ ]:
# Reduced number of options for each hyperparameter
n_estimators = [100, 200, 300]  # Fewer values for the number of trees
max_features = ['auto','sqrt']  #  # Number of features to consider at each split
max_depth = [10, 20, 30, 40, None]  # Fewer values for max depth
min_samples_split = [2, 5, 10]  # Keep essential options only
min_samples_leaf = [1, 2, 4]  # Reduced options for leaf samples
bootstrap = [True, False]  # Keep as is

# Create a lighter random grid
lighter_grid = {
    'n_estimators': n_estimators,
    'max_features': max_features,
    'max_depth': max_depth,
    'min_samples_split': min_samples_split,
    'min_samples_leaf': min_samples_leaf,
    'bootstrap': bootstrap
}

print(lighter_grid)

# Reduced number of iterations and cross-validation folds
# rf_random = RandomizedSearchCV(estimator=model_GBM, param_distributions=lighter_grid, n_iter=25, cv=3,
#                                verbose=2, random_state=42, n_jobs=-1)
rf_random = RandomizedSearchCV(estimator=model_RFR, param_distributions=lighter_grid, n_iter=25, cv=3,
                               verbose=2, random_state=42, n_jobs=-1)

# Fit the random search model
rf_random.fit(X_train, y_train)

In [ ]:
rf_random.best_estimator_

In [ ]:
def evaluate(model, test_features, y_test):
    predictions = model.predict(test_features)
    errors = abs(predictions - y_test)
    mae = 100 * np.mean(errors)
    print('Model Performance')
    print('Mean Absolute Error: {:0.4f}'.format(np.mean(errors)))
    # print('R^2: {:0.2f}'.format(metrics.r2_score(predictions, y_test)))
    return mae

#### Running base Model

In [ ]:
base_model = RandomForestRegressor(n_estimators = 100, random_state = 42)
base_model.fit(X_train, y_train)
base_accuracy = evaluate(base_model, X_val, y_val)

In [ ]:
best_random = rf_random.best_estimator_
random_accuracy = evaluate(best_random, X_val, y_val)

#### Comparisson

In [ ]:
print('Improvement of {:0.2f}%.'.format( 100 * (base_accuracy - random_accuracy) / base_accuracy))

Good improvment from the finetunuing when used with **RandomForestRegressor**, so we can try use the tinued model instead of basic

After we added the tuned Random Forest Regressor to all metrics table , we can see it is a little better from XGBoost Model

## Model Evaluation

### Performance Metrics

Performance Metrics with the fine tuned Model

In [ ]:
# make predictions on the validation set
y_val_pred_RFR_best_random = best_random.predict(X_val)

In [ ]:
model_dict = {"model":"RandomForest_Tuned"}
model_dict.update(regressionMetrics(y_val, y_val_pred_RFR_best_random))
new_row = pd.DataFrame([model_dict])
models_list = pd.concat([models_list, new_row], ignore_index=True)
models_list.sort_values('MAE')

### Validation with X_test, y_test

In [ ]:
models_ = {"LinearRegression":model_LR,
           "DecisionTree":model_DTR,
           "RandomForest":model_RFR,
           "ADABoost":model_ADABoost,
           "GBM":model_GBM,
           "SVR":model_SVR,
           "XGBoost":model_XGBoost,
           "RandomForest_Tuned":best_random
           }
models_eval_list = pd.DataFrame(columns= ['model', 'MAE', 'MSE', 'RMSE', 'RMSLE', 'R²'])

In [ ]:
for name, model in models_.items():
    print(model)
    y_test_pred = model.predict(X_test)
    # print(regressionMetrics(y_test, y_test_pred))
    model_dict = {"model":name}
    model_dict.update(regressionMetrics(y_test, y_test_pred))
    new_row = pd.DataFrame([model_dict])
    models_eval_list = pd.concat([models_eval_list, new_row], ignore_index=True)

In [ ]:
models_eval_list.sort_values('MAE')

## Store Objects States

In [ ]:
import joblib

joblib.dump(model_LR,"/content/drive/My Drive/Airbnb/model_LR.pkl")
joblib.dump(model_DTR,"/content/drive/My Drive/Airbnb/model_DTR.pkl")
joblib.dump(model_RFR,"/content/drive/My Drive/Airbnb/model_RFR.pkl")
joblib.dump(model_ADABoost,"/content/drive/My Drive/Airbnb/model_ADABoost.pkl")
joblib.dump(model_GBM,"/content/drive/My Drive/Airbnb/model_GBM.pkl")
joblib.dump(model_SVR,"/content/drive/My Drive/Airbnb/model_SVR.pkl")
joblib.dump(model_XGBoost,"/content/drive/My Drive/Airbnb/model_XGBoost.pkl")
joblib.dump(best_random,"/content/drive/My Drive/Airbnb/best_random.pkl")

models_list.to_pickle("/content/drive/My Drive/Airbnb/models_selection_list.pkl")
df.to_pickle("/content/drive/My Drive/Airbnb/df_Model_Selection_Post.pkl")

In [ ]:
# model_LR_ = joblib.load("/content/drive/My Drive/Airbnb/model_LR.pkl")